**Import and Load data**

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../Dataset/train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


**Cleaning**

In [2]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df = df.drop(columns=['Cabin'])

df_model = df.drop(columns=['PassengerId', 'Name', 'Ticket'])
df_model = pd.get_dummies(df_model, columns=['Sex', 'Embarked'], drop_first=True)
df_model.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S
0,0,3,22.0,1,0,7.2500,True,False,True
1,1,1,38.0,1,0,71.2833,False,False,False
2,1,3,26.0,0,0,7.9250,False,False,True
3,1,1,35.0,1,0,53.1000,False,False,True
4,0,3,35.0,0,0,8.0500,True,False,True


**Split data into trian/test**

In [3]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=['Survived'])
y = df_model['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

**Model Training**

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print("\nBaseline Classification Report:")
print(classification_report(y_test, y_pred_baseline, target_names=['Did Not Survive', 'Survived']))

Baseline Accuracy: 0.8101

Baseline Classification Report:
                 precision    recall  f1-score   support

Did Not Survive       0.83      0.86      0.84       105
       Survived       0.79      0.74      0.76        74

       accuracy                           0.81       179
      macro avg       0.81      0.80      0.80       179
   weighted avg       0.81      0.81      0.81       179



## Why Accuracy Alone Can Be Misleading

Accuracy only tells us the overall percentage of correct predictions, 
which can hide poor performance on the minority class in an imbalanced 
dataset. For example, if 90% of passengers did not survive, a model 
that always predicts "did not survive" would score 90% accuracy while 
never correctly identifying a single survivor. In the Titanic dataset, 
survivors are the minority class, so it's important to also look at 
recall for "Survived" specifically — this tells us how many actual 
survivors the model successfully identified, rather than just how 
often it was right overall. Precision, Recall, and F1-score give a 
fuller picture because they account for false positives and false 
negatives separately, rather than averaging everything into one number.

**Hyperparameter tuning with GridSearchCV**

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation F1 score: {grid_search.best_score_:.4f}")

Best parameters: {'C': 1, 'solver': 'liblinear'}
Best cross-validation F1 score: 0.7097


**Evaluate the tuned model**

In [6]:
tuned_model = grid_search.best_estimator_
y_pred_tuned = tuned_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, y_pred_tuned)
print(f"Tuned Accuracy: {tuned_accuracy:.4f}")
print("\nTuned Classification Report:")
print(classification_report(y_test, y_pred_tuned, target_names=['Did Not Survive', 'Survived']))

Tuned Accuracy: 0.7821

Tuned Classification Report:
                 precision    recall  f1-score   support

Did Not Survive       0.79      0.85      0.82       105
       Survived       0.76      0.69      0.72        74

       accuracy                           0.78       179
      macro avg       0.78      0.77      0.77       179
   weighted avg       0.78      0.78      0.78       179



**Before/after comparison table**

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (Survived)', 'Recall (Survived)', 'F1-score (Survived)'],
    'Baseline': [
        accuracy_score(y_test, y_pred_baseline),
        precision_score(y_test, y_pred_baseline),
        recall_score(y_test, y_pred_baseline),
        f1_score(y_test, y_pred_baseline)
    ],
    'Tuned': [
        accuracy_score(y_test, y_pred_tuned),
        precision_score(y_test, y_pred_tuned),
        recall_score(y_test, y_pred_tuned),
        f1_score(y_test, y_pred_tuned)
    ]
})

comparison['Improvement'] = comparison['Tuned'] - comparison['Baseline']
comparison

,Metric,Baseline,Tuned,Improvement
0,Accuracy,0.810056,0.782123,-0.027933
1,Precision (Survived),0.785714,0.761194,-0.024520
2,Recall (Survived),0.743243,0.689189,-0.054054
3,F1-score (Survived),0.763889,0.723404,-0.040485
